# Deep Learning (DLY0100) - Sección 012V
## Perceptrón Multicapa (MLP) para clasificación de fracturas óseas

**Evaluación:** Evaluación Parcial 1

**Integrantes:** Luis Muñoz, Aran Opazo

**Fecha:** 3 de Octubre, 2026

> Proyecto académico: el modelo no está pensado para uso clínico ni para apoyar diagnósticos reales.

### Estructura del notebook

1. Introducción
2. Contexto
3. Problema
4. Entorno
5. Datos
6. Preprocesamiento
7. División del dataset
8. Experimentos (entrenamiento y validación)
9. Modelo final
10. Evaluación del mejor modelo
11. Visualización de predicciones
12. Conclusiones y trabajos futuros
13. Reproducción y referencias

## 1. Introducción

En este trabajo se entrena un Perceptrón Multicapa (MLP) para clasificar radiografías según el tipo de fractura ósea, usando el dataset Bone Break Classification el cual contiene 1.129 radiografías de 10 tipos de fractura distintos. La exploración de los datos mostró que hay clases con más imágenes que otras, hay imágenes de tamaños muy distintos e imágenes duplicadas. Para el propósito de este trabajo, las radiografías se convirtieron a escala de grises, se redimensionaron, se escalaron entre 0 y 1 y se aplanaron en vectores. Finalmente, se dividieron en entrenamiento, validación y test.

Partiendo de un modelo base, el modelo fue tomando su forma final con un método iterativo: en cada paso se cambió un solo aspecto (cantidad de neuronas, cantidad de capas, tasa de aprendizaje, balance de clases, batch size, función de activación, inicialización de pesos, función de pérdida, regularización y optimizador), y se mantuvo el cambio solo si mejoraba el desempeño en validación sin sobreajuste. El modelo final se reentrenó con más épocas, guardando el mejor modelo, y se evaluó una sola vez en test.

**Resultado:** [COMPLETAR al final: arquitectura y configuración del modelo final, y su F1 macro y accuracy en test].

**Conclusión principal:** [COMPLETAR al final: qué decisiones aportaron más y cuál es la principal limitación del MLP para este problema].

## 2. Contexto

Las fracturas óseas se diagnostican mediante radiografías principalmente. Es muy importante, además de detectar la fractura, identificar su tipo (por ejemplo, si el hueso se rompió en varios fragmentos, en espiral o solo con una fisura), porque de ello dependerá el tratamiento: inmovilización, cirugía, etc.

- **Dataset:** Bone Break Classification Image Dataset (Kaggle).
- **Contenido:** radiografías de 10 tipos de fractura: avulsión, conminuta, fractura-luxación, en tallo verde, fisura, impactada, longitudinal, oblicua, patológica y espiral.

## 3. Problema

Identificar el tipo de fractura en una radiografía requiere experiencia y tiempo, y un error de clasificación puede llevar a un tratamiento inadecuado. A partir de las diferentes imagenes, se busca determinar a cuál de los 10 tipos de fractura corresponde cada una de ellas. Cada radiografía pertenece a un solo tipo, por lo que se trata de una clasificación multiclase con clases excluyentes.

**Objetivos:**
- Entrenar un MLP con TensorFlow/Keras que clasifique el tipo de fractura.
- Mejorarlo de forma iterativa, justificando cada decisión con resultados en validación.
- Evaluar el modelo final con accuracy, precision, recall y F1-score, poniendo atención en el recall ya que no reconocer un tipo de fractura se considera más grave que una falsa alarma.

## 4. Entorno

### 4.1 GPU y librerías

El entrenamiento se ejecuta en Google Colab utilizando las librerías que se detallan:

| Librería | Para qué se usa |
|---|---|
| NumPy / Pandas | Arreglos de números (imágenes vectorizadas) y tablas de resultados |
| Matplotlib / Seaborn | Generación de Gráficos |
| PIL (Pillow) | Lectura y transformación de imágenes (incluida la data augmentation) |
| scikit-learn | División estratificada de los datos y cálculo de métricas |
| TensorFlow / Keras | Construcción y entrenamiento del MLP |

In [2]:
import os, sys, json, time, random, hashlib, platform, datetime, zipfile
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # oculta mensajes informativos de TensorFlow

# Si el notebook se ejecuta desde la carpeta notebooks/, se sube a la raíz del proyecto
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             log_loss, confusion_matrix, classification_report)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

tf.get_logger().setLevel("ERROR")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

print(f"Python {platform.python_version()} | TensorFlow {tf.__version__} | Keras {keras.__version__} | "
      f"NumPy {np.__version__} | Pandas {pd.__version__}")

Python 3.13.15 | TensorFlow 2.20.0 | Keras 3.13.2 | NumPy 2.1.3 | Pandas 2.2.3


Finalmente se muestran las versiones de las herramientas utilizadas.

### 4.2 Semilla

Se fija la semilla en 42 y se activa el modo determinista de TensorFlow, para que se entreguen siempre los mismos resultados. Además, la función reiniciar_semillas() vuelve a fijar la semilla antes de cada entrenamiento, para que así la diferencia entre dos experimentos se deba solo al parámetro que se haya cambiado.

In [3]:
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
tf.config.experimental.enable_op_determinism()   # para repetir los mismios resultados

def reiniciar_semillas(seed=None):
    """Restablece todas las fuentes de aleatoriedad (Python, NumPy y TensorFlow)."""
    seed = SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

reiniciar_semillas()
print("Semilla fija:", SEED)

Semilla fija: 42


### 4.3 Carpetas del proyecto

Se crean las carpetas donde se guardan los datos descargados (data), los modelos (models), los gráficos (images) y las tablas de resultados (results).

In [4]:
for carpeta in ["data", "models", "images", "results"]:
    os.makedirs(carpeta, exist_ok=True)
print("Contenido del directorio de trabajo:", sorted(os.listdir(".")))

Contenido del directorio de trabajo: ['.config', 'data', 'images', 'models', 'results', 'sample_data']


## 5. Datos

### 5.1 Descarga

Se descarga el dataset Bone Break Classification directamente desde Kaggle.

In [5]:
KAGGLE_DATASET = "pkdarabi/bone-break-classification-image-dataset"
RUTA_ZIP = "data/bone-break-classification-image-dataset.zip"
RUTA_RAW = "data/bone_raw"          # carpeta donde se descomprime el dataset

if os.path.isdir(RUTA_RAW) and os.listdir(RUTA_RAW):
    print("El dataset ya está descomprimido en", RUTA_RAW)
else:
    # Credenciales de Kaggle desde secrets
    try:
        from google.colab import userdata
        token = userdata.get("KAGGLE_TOKEN")
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
            f.write(token)
        os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
    except ImportError:
        pass   # ejecución local: se espera el token en ~/.kaggle/
    except Exception as e:
        raise RuntimeError("No se encontró el secreto KAGGLE_TOKEN. Agrégalo en el panel 🔑 de Colab "
                           "y habilita el acceso para este notebook.") from e

    # 2) Descarga y descompresión
    codigo = os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p data -q")
    if codigo != 0 or not os.path.exists(RUTA_ZIP):
        raise RuntimeError("Falló la descarga desde Kaggle: revisa el token y la conexión.")
    with zipfile.ZipFile(RUTA_ZIP) as z:
        z.extractall(RUTA_RAW)
    print("Dataset descargado y descomprimido en", RUTA_RAW)

Dataset descargado y descomprimido en data/bone_raw


### 5.2 Inventario e integridad de las imágenes

Al descargar el dataset, las imágenes se organizan primero por clase y luego en train o test.

También se verifica que cada archivo se pueda abrir y no esté vacío ni dañado, se registran su tamaño y modo de color, y se calcula un hash MD5 que permite detectar imágenes duplicadas.

In [6]:
EXTENSIONES = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp")

def encontrar_raiz(ruta):
    """Busca la carpeta cuyas subcarpetas (las clases) contienen una carpeta Train."""
    for actual, subcarpetas, _ in os.walk(ruta):
        if subcarpetas and all(
            any(s.lower() == "train" for s in os.listdir(os.path.join(actual, c))) for c in subcarpetas
        ):
            return actual
    raise FileNotFoundError(f"No se encontró la estructura <clase>/Train dentro de {ruta}")

RAIZ = encontrar_raiz(RUTA_RAW)
CLASES = sorted(d for d in os.listdir(RAIZ) if os.path.isdir(os.path.join(RAIZ, d)))
N_CLASES = len(CLASES)

filas = []
for clase in CLASES:
    for sub in os.listdir(os.path.join(RAIZ, clase)):
        particion = sub.lower()                        # "train" o "test"
        if particion not in ("train", "test"):
            continue
        carpeta = os.path.join(RAIZ, clase, sub)
        for archivo in sorted(os.listdir(carpeta)):
            if not archivo.lower().endswith(EXTENSIONES):
                continue
            ruta = os.path.join(carpeta, archivo)
            fila = {"ruta": ruta, "clase": clase, "particion": particion, "bytes": os.path.getsize(ruta),
                    "ok": False, "ancho": None, "alto": None, "modo": None, "hash": None}
            try:
                with open(ruta, "rb") as fh:
                    fila["hash"] = hashlib.md5(fh.read()).hexdigest()
                with Image.open(ruta) as im:
                    im.verify()                         # detecta archivos dañados
                with Image.open(ruta) as im:
                    fila["ancho"], fila["alto"], fila["modo"] = im.size[0], im.size[1], im.mode
                fila["ok"] = fila["bytes"] > 0
            except Exception:
                pass                                    # queda marcado como ok = False
            filas.append(fila)

df_inv = pd.DataFrame(filas)
print("Carpeta raíz del dataset:", RAIZ)
print(f"{N_CLASES} clases:", CLASES)
print("\nImágenes por partición original:\n", df_inv.groupby("particion").size().to_string())
print("\nArchivos vacíos o dañados:", int((~df_inv["ok"]).sum()))
display(df_inv.head())

Carpeta raíz del dataset: data/bone_raw/Bone Break Classification/Bone Break Classification
10 clases: ['Avulsion fracture', 'Comminuted fracture', 'Fracture Dislocation', 'Greenstick fracture', 'Hairline Fracture', 'Impacted fracture', 'Longitudinal fracture', 'Oblique fracture', 'Pathological fracture', 'Spiral Fracture']

Imágenes por partición original:
 particion
test     140
train    989

Archivos vacíos o dañados: 0


,ruta,clase,particion,bytes,ok,ancho,alto,modo,hash
0,data/bone_raw/Bone Break Classification/Bone B...,Avulsion fracture,train,31581,True,205,586,RGB,5eafa7eee6488dc70db59991d06dbc1e
1,data/bone_raw/Bone Break Classification/Bone B...,Avulsion fracture,train,33304,True,172,585,RGB,0348b50e9ca152e082da4f5e4a07124a
2,data/bone_raw/Bone Break Classification/Bone B...,Avulsion fracture,train,20918,True,320,640,RGB,dc0f1b85917728bd4be4fa4943aa5633
3,data/bone_raw/Bone Break Classification/Bone B...,Avulsion fracture,train,20918,True,320,640,RGB,dc0f1b85917728bd4be4fa4943aa5633
4,data/bone_raw/Bone Break Classification/Bone B...,Avulsion fracture,train,18008,True,245,640,RGB,8bb44e3327a09e20e9488111766d0e96


> La salida del código anterior indica que el dataset se leyó correctamente ya que se encontraron las 10 clases de fractura, con 989 imágenes en las carpetas Train y 140 en las carpetas Test. Ningún archivo está vacío ni dañado, por lo que no será necesario excluir imágenes por problemas de lectura.

### 5.3 Cardinalidad de los datos

Un resumen de la cantidad de imágenes, categorías a clasificar, píxeles por imagen y balance de clases. Los tamaños y el balance se calculan solo con las imágenes de entrenamiento.

In [7]:
train_ok = df_inv[df_inv["ok"] & (df_inv["particion"] == "train")]
n_test = int((df_inv["ok"] & (df_inv["particion"] == "test")).sum())
pixeles = train_ok["ancho"] * train_ok["alto"]
por_clase = train_ok["clase"].value_counts()

cardinalidad = pd.Series({
    "categorías a clasificar": N_CLASES,
    "imágenes en Train": len(train_ok),
    "imágenes en Test": n_test,
    "imágenes por clase en Train": f"{por_clase.min()} a {por_clase.max()}",
    "razón de desbalance (Train)": f"{por_clase.max() / por_clase.min():.2f}",
    "píxeles por imagen original (Train)": f"{int(pixeles.min()):,} a {int(pixeles.max()):,} (mediana {int(pixeles.median()):,})",
    "canales de color originales": ", ".join(train_ok["modo"].unique()),
    "entradas del MLP tras el preprocesamiento": "64 × 64 × 1 = 4.096 (escala de grises, 64×64)",
}, name="valor").to_frame()
display(cardinalidad)

,valor
categorías a clasificar,10
imágenes en Train,989
imágenes en Test,140
imágenes por clase en Train,68 a 137
razón de desbalance (Train),2.01
píxeles por imagen original (Train),"9,280 a 409,600 (mediana 213,760)"
canales de color originales,RGB
entradas del MLP tras el preprocesamiento,"64 × 64 × 1 = 4.096 (escala de grises, 64×64)"


> **Interpretación (completar después de ejecutar)**

### 5.4 Exploración de los datos

Cada análisis termina con una conclusión que conecta lo observado con una decisión de preprocesamiento o de modelado.

**Solo se explora el conjunto de entrenamiento.** Las imágenes de Test deben ser completamente nuevas para el modelo hasta la evaluación final (sección 10); si se analizaran aquí (su distribución, tamaños o brillo), las decisiones de preparación estarían influidas por datos que deberían ser desconocidos.

#### 5.4.1 Distribución de clases (balance)

Para medir el balance se usa la razón de desbalance: imágenes de la clase más numerosa ÷ imágenes de la menos numerosa. Si la razón es cercana a 1, las clases están balanceadas; si es 2, una clase tiene el doble de ejemplos que otra y el modelo tenderá a favorecer a las más numerosas.

In [ ]:
# Imágenes por clase en Train, ordenadas de menor a mayor
tr = df_inv[df_inv["ok"] & (df_inv["particion"] == "train")]
conteo = tr["clase"].value_counts().sort_values()
proporcion = conteo / conteo.sum()
colores = ["#c62828" if v == conteo.min() else "#2e7d32" if v == conteo.max() else "steelblue" for v in conteo]

fig, ax = plt.subplots(figsize=(10, 5))
barras = ax.barh(conteo.index, conteo.values, color=colores)
ax.axvline(conteo.mean(), ls="--", c="k", label=f"promedio ({conteo.mean():.0f} imágenes)")
for barra, v, p in zip(barras, conteo.values, proporcion.values):
    ax.text(v + 1, barra.get_y() + barra.get_height() / 2, f"{v} ({p:.1%})", va="center", fontsize=9)
ax.set_xlabel("imágenes en Train")
ax.set_title(f"Desbalance de clases en Train (razón más numerosa / menos numerosa = {conteo.max() / conteo.min():.2f})")
ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig("images/distribucion_clases.png", dpi=150, bbox_inches="tight"); plt.show()

print(f"Clase más numerosa: {conteo.idxmax()} ({conteo.max()}) | menos numerosa: {conteo.idxmin()} ({conteo.min()}) | "
      f"razón de desbalance: {conteo.max() / conteo.min():.2f}")

> **Interpretación (completar después de ejecutar)**

#### 5.4.2 Ejemplos de cada tipo de fractura

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(4, 5, figsize=(17, 14))
for i, clase in enumerate(CLASES):
    rutas = df_inv[(df_inv["clase"] == clase) & df_inv["ok"] & (df_inv["particion"] == "train")]["ruta"].values
    for j, ruta in enumerate(rng.choice(rutas, size=2, replace=False)):   # 2 ejemplos al azar por clase
        ax = axes.ravel()[2 * i + j]
        ax.imshow(Image.open(ruta).convert("L"), cmap="gray")
        ax.set_title(clase, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.savefig("images/ejemplos_por_clase.png", dpi=120, bbox_inches="tight"); plt.show()

> **Interpretación (completar después de ejecutar)**

#### 5.4.3 Tamaños y modos de color

Un MLP recibe un vector de largo fijo, así que todas las imágenes deben llevarse al mismo tamaño. Aquí se revisa, en las imágenes de entrenamiento, cuánto varían los tamaños originales y en qué modo de color vienen (RGB = color, L = escala de grises).

In [ ]:
ok = df_inv[df_inv["ok"] & (df_inv["particion"] == "train")]
print("Modos de color:", ok["modo"].value_counts().to_dict())
print(f"Tamaños distintos: {ok.groupby(['ancho', 'alto']).ngroups}")
print(f"Ancho: {ok['ancho'].min():.0f} a {ok['ancho'].max():.0f} px (mediana {ok['ancho'].median():.0f})")
print(f"Alto : {ok['alto'].min():.0f} a {ok['alto'].max():.0f} px (mediana {ok['alto'].median():.0f})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(ok["ancho"], ok["alto"], s=8, alpha=0.4)
ax[0].set_xlabel("ancho (px)"); ax[0].set_ylabel("alto (px)"); ax[0].set_title("Tamaño original de cada imagen")
ax[1].hist(ok["ancho"] / ok["alto"], bins=40)
ax[1].set_xlabel("ancho / alto"); ax[1].set_title("Relación de aspecto (1 = cuadrada)")
plt.tight_layout(); plt.savefig("images/tamanos_originales.png", dpi=150); plt.show()

> **Interpretación (completar después de ejecutar)**

#### 5.4.4 Duplicados y posible fuga de información

Se revisan imágenes repetidas por dos razones:

1. **Fuga de información:** si una misma radiografía estuviera en entrenamiento y en test, el modelo sería evaluado con una imagen que ya vio y las métricas quedarían engañosamente altas.
2. **Peso duplicado:** una imagen repetida dentro de entrenamiento cuenta dos veces al aprender, dándole más importancia que al resto.

Se usan los hash calculados en el inventario: dos archivos con el mismo hash son idénticos. Es la única revisión que consulta las carpetas Test, y solo para comparar huellas digitales, sin analizar su contenido.

In [ ]:
ok = df_inv[df_inv["ok"]]
print("Duplicados exactos dentro de una misma partición:", int(ok.duplicated(["particion", "hash"]).sum()))
print("Duplicados con etiquetas contradictorias:", int((ok.groupby("hash")["clase"].nunique() > 1).sum()))
h_train = set(ok[ok["particion"] == "train"]["hash"])
h_test = set(ok[ok["particion"] == "test"]["hash"])
print("Imágenes idénticas presentes en train y en test (fuga):", len(h_train & h_test))

> Según la salida del bloque de código anterior, se encontraron 3 imágenes duplicadas dentro de una misma partición (en Train). Ninguna aparece con dos etiquetas distintas, y ninguna imagen de test está repetida en train, por lo que no hay fuga de información entre entrenamiento y prueba.
>
> En base a lo encontrado, se decide conservar una sola copia de cada duplicado, para que ninguna imagen pese el doble durante el entrenamiento; así, las 989 imágenes de Train quedan en 986.
>
> Consideraciones de limitaciones: este control solo detecta copias idénticas; una misma radiografía rotada o recortada no se detectaría.

#### 5.4.5 Brillo y contraste

Las radiografías pueden variar mucho en exposición. Se compara el brillo medio (promedio de los píxeles) y el contraste (desviación estándar de los píxeles) entre clases, sobre una muestra de entrenamiento.

In [ ]:
muestra = df_inv[df_inv["ok"] & (df_inv["particion"] == "train")].sample(frac=1, random_state=SEED) \
            .groupby("clase").head(60).copy()
pix = [np.asarray(Image.open(r).convert("L"), dtype=np.float32) for r in muestra["ruta"]]
muestra["brillo"] = [p.mean() for p in pix]
muestra["contraste"] = [p.std() for p in pix]

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sns.boxplot(data=muestra, x="clase", y="brillo", ax=ax[0]); ax[0].set_title("Brillo medio por clase")
sns.boxplot(data=muestra, x="clase", y="contraste", ax=ax[1]); ax[1].set_title("Contraste por clase")
for a in ax:
    a.tick_params(axis="x", rotation=60); a.set_xlabel("")
plt.tight_layout(); plt.savefig("images/brillo_contraste.png", dpi=150); plt.show()

q1, q3 = muestra["brillo"].quantile([0.25, 0.75]); iqr = q3 - q1
atipicas = ((muestra["brillo"] < q1 - 1.5 * iqr) | (muestra["brillo"] > q3 + 1.5 * iqr)).sum()
print(f"Imágenes con brillo atípico (regla 1,5·IQR): {atipicas} de {len(muestra)}")

> **Interpretación (completar después de ejecutar)**

#### 5.4.6 Dificultades esperadas para un MLP

- **Pérdida de la estructura espacial:** el MLP aplana la imagen en un vector y trata cada píxel como una variable independiente; no sabe qué píxeles son vecinos. Las diferencias entre tipos de fractura (la dirección de una línea, si el hueso está fragmentado) son justamente patrones espaciales.
- **Detalles finos:** una fisura (hairline) es una línea muy delgada que puede desaparecer al reducir la resolución.
- **Pocas imágenes para muchas clases:** unas 100 imágenes por clase en entrenamiento; con miles de entradas por imagen, el riesgo de sobreajuste es alto.
- **Variabilidad:** distintas zonas del cuerpo, encuadres y exposiciones dentro de una misma clase.

## 6. Preprocesamiento

Cada radiografía se transforma en la entrada que admite un MLP:

1. **Escala de grises:** las radiografías no tienen información de color; en RGB cada píxel se repetiría 3 veces, triplicando las entradas sin aportar información.
2. **Redimensionar** a un tamaño cuadrado fijo, porque el MLP necesita que todas las entradas tengan la misma cantidad de valores. Se preparan 32×32, 64×64 y 128×128 para compararlos en la sección 8.5.
3. **Escalar los píxeles entre 0 y 1**, dividiendo por 255. Como se vio en clases, con entradas en la misma escala el descenso del gradiente llega al mínimo de forma más directa. Como 255 es un valor fijo, no depende de los datos y no genera fuga de información.
4. **Flatten (aplanar):** la imagen se convierte en un vector (por ejemplo, 64×64 → 4.096 valores).
5. **One-hot encoding** de la etiqueta: cada clase se convierte en un vector de 10 posiciones con un 1 en la posición de su clase. Es el formato que compara la entropía cruzada con la salida softmax.

Aquí se definen las funciones y se muestra un ejemplo; se aplican a cada partición en la sección 7.2, después de dividir el dataset.

In [ ]:
TAMANOS = [32, 64, 128]

def cargar_imagenes(rutas, tamano):
    """Lee cada imagen, la pasa a escala de grises y la redimensiona a tamano x tamano."""
    X = np.empty((len(rutas), tamano, tamano), dtype=np.uint8)
    for i, ruta in enumerate(rutas):
        with Image.open(ruta) as im:
            X[i] = np.asarray(im.convert("L").resize((tamano, tamano), Image.BILINEAR))
    return X

def preprocesar(imagenes):
    """Aplana cada imagen en un vector (flatten) y escala los píxeles de 0-255 a 0-1."""
    return imagenes.reshape(len(imagenes), -1).astype("float32") / 255.0

indice = {clase: i for i, clase in enumerate(CLASES)}   # clase → número (0 a 9)

# Ejemplo con una radiografía de entrenamiento
ejemplo = df_inv[df_inv["ok"] & (df_inv["particion"] == "train")].iloc[0]
original = Image.open(ejemplo["ruta"]).convert("L")
fig, ax = plt.subplots(1, 5, figsize=(20, 4))
ax[0].imshow(original, cmap="gray"); ax[0].set_title(f"Original {original.size[0]}×{original.size[1]}")
for a, t in zip(ax[1:4], TAMANOS):
    a.imshow(cargar_imagenes([ejemplo["ruta"]], t)[0], cmap="gray"); a.set_title(f"{t}×{t} → {t * t} entradas")
for a in ax[:4]:
    a.axis("off")
vector = preprocesar(cargar_imagenes([ejemplo["ruta"]], 64))[0]
ax[4].hist(vector, bins=40); ax[4].set_title(f"Vector aplanado y escalado ({vector.size} valores entre 0 y 1)")
plt.suptitle(f"Clase: {ejemplo['clase']}"); plt.tight_layout()
plt.savefig("images/antes_despues_preprocesamiento.png", dpi=130); plt.show()

display(pd.DataFrame({"índice": range(N_CLASES), "clase": CLASES}))
one_hot = keras.utils.to_categorical([indice[ejemplo["clase"]]], N_CLASES)[0].astype(int)
print(f"One-hot de '{ejemplo['clase']}': índice {indice[ejemplo['clase']]} → {one_hot}")

> **Interpretación (completar después de ejecutar)**

## 7. División del dataset

| Conjunto | Para qué se usa |
|---|---|
| **Entrenamiento** | Ajustar los **parámetros** del modelo (pesos y bias): es lo que optimiza el descenso del gradiente. |
| **Validación** | Ajustar los **hiperparámetros** (neuronas, capas, tasa de aprendizaje, etc.) y controlar la generalización durante el entrenamiento (early stopping). |
| **Test** | **Evaluación final** del modelo, una sola vez, con imágenes nunca vistas. |

### 7.1 Entrenamiento, validación y test

- **Test:** las imágenes de las carpetas `Test` del dataset, sin tocarlas hasta la sección 10.
- **Validación:** 20% de las imágenes de `Train`, separado de forma **aleatoria y estratificada** (cada clase conserva su proporción).
- **Entrenamiento:** el 80% restante de `Train`.
- Se excluyen los archivos dañados, los duplicados exactos y las imágenes de `Train` idénticas a alguna de `Test`.

In [ ]:
VAL_SIZE = 0.20

base = df_inv[df_inv["ok"]].drop_duplicates(["particion", "hash"], keep="first")
test_df = base[base["particion"] == "test"]
resto = base[(base["particion"] == "train") & ~base["hash"].isin(set(test_df["hash"]))]
train_df, val_df = train_test_split(resto, test_size=VAL_SIZE, stratify=resto["clase"], random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]

resumen = pd.DataFrame({"entrenamiento": train_df["clase"].value_counts(),
                        "validación": val_df["clase"].value_counts(),
                        "test": test_df["clase"].value_counts()}).fillna(0).astype(int)
resumen.loc["TOTAL"] = resumen.sum()
display(resumen)

> De las 989 imágenes de `Train` se eliminaron 3 duplicados exactos (sección 5.4.4). Las 986 restantes se dividieron de forma estratificada en **788 de entrenamiento (80%)** y **198 de validación (20%)**, y las **140 imágenes de `Test`** se mantienen intactas para la evaluación final. Como el test tiene solo unas 14 imágenes por clase, un solo error cambia el recall de esa clase en unos 7 puntos porcentuales, así que sus métricas deben leerse con cautela.

### 7.2 Aplicación del preprocesamiento

Se aplica el preprocesamiento de la sección 6 a cada partición, en las tres resoluciones, y se codifican las etiquetas en one-hot.

In [ ]:
y_train = train_df["clase"].map(indice).values
y_val   = val_df["clase"].map(indice).values
y_test  = test_df["clase"].map(indice).values
Y_train, Y_val, Y_test = (keras.utils.to_categorical(y, N_CLASES) for y in (y_train, y_val, y_test))

DATOS = {}
for t in TAMANOS:
    imagenes = [cargar_imagenes(d["ruta"].values, t) for d in (train_df, val_df, test_df)]
    DATOS[t] = {"img": imagenes, "X": [preprocesar(a) for a in imagenes]}
    print(f"{t}x{t}: {DATOS[t]['X'][0].shape[1]:>6} entradas por imagen | entrenamiento {DATOS[t]['X'][0].shape} | "
          f"validación {DATOS[t]['X'][1].shape} | test {DATOS[t]['X'][2].shape}")

> La salida muestra la cantidad de entradas que recibe el MLP según la resolución, con la forma (cantidad de imágenes, entradas por imagen):
>
> | Resolución | Entradas por imagen | Entrenamiento | Validación | Test |
> |---|---|---|---|---|
> | 32×32 | 1.024 | 788 | 198 | 140 |
> | 64×64 | 4.096 | 788 | 198 | 140 |
> | 128×128 | 16.384 | 788 | 198 | 140 |
>
> Cada vez que el lado de la imagen se duplica, las entradas se multiplican por 4, y con ellas los pesos de la primera capa (entradas × neuronas). Con solo 788 imágenes de entrenamiento, más entradas conservan más detalle, pero también dan más parámetros con los que memorizar (sobreajuste). Este equilibrio se mide en la sección 8.5.

### 7.3 Conjunto de entrenamiento balanceado + data augmentation

Como las clases están desbalanceadas (sección 5.4.1), se prepara una **segunda versión del conjunto de entrenamiento**, que se comparará con la original en la sección 8.8:

1. **Submuestreo:** se toma la misma cantidad de imágenes de cada clase, igual a la de la clase menos numerosa del conjunto de entrenamiento. En la carpeta `Train` la clase más pequeña tiene 68 imágenes; después de separar la validación, en entrenamiento le quedan menos, y ese es el número que se usa para no repetir imágenes.
2. **Data augmentation:** de cada imagen seleccionada se generan 2 variantes con transformaciones realistas para radiografías: volteo horizontal, rotación de hasta ±10° y cambios de brillo y contraste de hasta ±15%. Así se recupera cantidad de imágenes y se agrega variedad.

**Reglas importantes:**
- La **validación y el test no se modifican**: deben representar radiografías reales, con su distribución original. Además, ambas versiones del entrenamiento se comparan contra **la misma validación**.
- El aumento se aplica **después** de separar las particiones, para que variantes de una misma radiografía no terminen en entrenamiento y validación a la vez (fuga de información).

**Alternativa: agrupar por zona del cuerpo.** se consideró agrupar las fracturas en parte superior e inferior del cuerpo, pero el dataset no incluye esa información: las carpetas indican el tipo de fractura, no la zona. Aplicarla requeriría etiquetar a mano las 1.129 radiografías y cambiaría el problema planteado (de "tipo de fractura" a "zona del cuerpo"), por lo que se deja como alternativa futura.

In [ ]:
# 1. Submuestreo: misma cantidad de imágenes por clase
N_POR_CLASE = int(train_df["clase"].value_counts().min())
bal_df = train_df.groupby("clase", group_keys=False).sample(n=N_POR_CLASE, random_state=SEED).reset_index(drop=True)
y_bal = bal_df["clase"].map(indice).values

# 2. Data augmentation: transformaciones realistas para radiografías
N_VARIANTES = 2

def aumentar(img, rng):
    """Genera una variante de una radiografía (arreglo en escala de grises)."""
    im = Image.fromarray(img)
    if rng.random() < 0.5:
        im = im.transpose(Image.FLIP_LEFT_RIGHT)                          # volteo horizontal
    im = im.rotate(rng.uniform(-10, 10), resample=Image.BILINEAR)         # rotación de hasta ±10°
    im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.85, 1.15))     # brillo ±15%
    im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.85, 1.15))       # contraste ±15%
    return np.asarray(im)

DATOS_BAL = {}
for t in TAMANOS:
    rng_aug = np.random.default_rng(SEED)
    originales = cargar_imagenes(bal_df["ruta"].values, t)
    variantes = [np.stack([aumentar(im, rng_aug) for im in originales]) for _ in range(N_VARIANTES)]
    imagenes = np.concatenate([originales] + variantes)
    DATOS_BAL[t] = {"img": imagenes, "X": preprocesar(imagenes)}

y_train_bal = np.tile(y_bal, N_VARIANTES + 1)
Y_train_bal = keras.utils.to_categorical(y_train_bal, N_CLASES)

print(f"Imágenes por clase tras el submuestreo: {N_POR_CLASE} ({len(bal_df)} en total)")
print(f"Con {N_VARIANTES} variantes por imagen: {len(y_train_bal)} imágenes de entrenamiento "
      f"({N_POR_CLASE * (N_VARIANTES + 1)} por clase)")
print(f"Entrenamiento original: {len(y_train)} imágenes (desbalanceado) | validación: {len(y_val)} (sin cambios)")

In [ ]:
# Distribución antes y después del balanceo
antes = pd.Series(y_train).map(dict(enumerate(CLASES))).value_counts().reindex(CLASES)
despues = pd.Series(y_train_bal).map(dict(enumerate(CLASES))).value_counts().reindex(CLASES)
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(N_CLASES)
ax.bar(x - 0.2, antes.values, width=0.4, label="original (desbalanceado)", color="#9e9e9e")
ax.bar(x + 0.2, despues.values, width=0.4, label="balanceado + data augmentation", color="steelblue")
ax.set_xticks(x); ax.set_xticklabels(CLASES, rotation=45, ha="right")
ax.set_ylabel("imágenes de entrenamiento"); ax.set_title("Conjunto de entrenamiento antes y después del balanceo")
ax.legend(); plt.tight_layout(); plt.savefig("images/balanceo_clases.png", dpi=150, bbox_inches="tight"); plt.show()

# Ejemplos: una radiografía y sus variantes generadas
n = len(bal_df)
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for fila_ax, i in zip(axes, [0, n // 2]):
    for a, j, titulo in zip(fila_ax, [i, i + n, i + 2 * n], ["original", "variante 1", "variante 2"]):
        a.imshow(DATOS_BAL[64]["img"][j], cmap="gray"); a.set_title(f"{bal_df['clase'][i]}\n{titulo}", fontsize=9)
        a.axis("off")
plt.tight_layout(); plt.savefig("images/ejemplos_data_augmentation.png", dpi=130, bbox_inches="tight"); plt.show()

> **Interpretación (completar después de ejecutar)**

### 7.4 Aislamiento del test

Desde este punto, las imágenes de test **no se usan** para entrenar, comparar ni elegir ninguna configuración. Todas las decisiones de la sección 8 se toman con **validación**. El test se evalúa **una sola vez**, en la sección 10, con el modelo final ya cerrado.

## 8. Experimentos (entrenamiento y validación)

### 8.1 Métricas: qué significan y cómo se calculan

Las métricas se calculan para cada tipo de fractura por separado. Por ejemplo, para la clase fractura espiral: VP = fracturas espirales que el modelo predijo como espirales; FP = fracturas de otros tipos que el modelo predijo como espirales; FN = fracturas espirales que el modelo predijo como otro tipo. Luego se repite el cálculo con cada una de las 10 clases.

| Métrica | Fórmula | Qué responde (ejemplo con fractura espiral) |
|---|---|---|
| **Accuracy** | aciertos / total | ¿Qué proporción de todas las radiografías se clasificó bien? Engañosa con clases desbalanceadas. |
| **Precision** | VP / (VP + FP) | De las que el modelo dijo "fractura espiral", ¿cuántas lo eran? |
| **Recall** | VP / (VP + FN) | De las fracturas espirales reales, ¿cuántas encontró el modelo? |
| **F1** | 2·P·R / (P + R) | Media armónica: solo es alta si precision **y** recall lo son. |

- **Promedio macro:** se calcula la métrica en cada clase y se promedian las 10 sin ponderar, de modo que cada tipo de fractura pesa lo mismo aunque tenga menos imágenes. Por el desbalance entre clases observado en la sección 5.4.1, la métrica de selección es el F1 macro.
- **Entropía cruzada de validación (log-loss):** es la misma entropía cruzada que usa Keras como función de pérdida, calculada igual para todos los experimentos. Se usa como desempate porque la val_loss de Keras no es comparable entre experimentos entrenados con distintas funciones de pérdida.
- **En este caso,** el recall es especialmente relevante: no reconocer un tipo de fractura (por ejemplo, una fractura patológica, asociada a enfermedades del hueso) es más grave que una falsa alarma. **[COMPLETAR: qué clase o error consideramos más crítico y por qué].**

### 8.2 Herramientas del experimento

Para no copiar el modelo en cada experimento, se usan funciones:
- `construir_mlp(...)` arma el MLP a partir de parámetros: capas y neuronas, activación, dropout, batch normalization, optimizador, tasa de aprendizaje y función de pérdida.
- `ejecutar_experimento(...)` parte del **modelo actual**, aplica un cambio, lo entrena, lo evalúa en validación y registra los resultados.
- `elegir(...)` escoge el mejor experimento **sin sobreajuste**, y `avanzar(...)` lo convierte en el nuevo modelo actual.
- `graficar(...)` muestra las curvas de **entrenamiento (línea continua) y validación (línea punteada)** de varios experimentos.

In [ ]:
def arq(texto):
    """'2x30' → 2 capas ocultas de 30 neuronas; '8-4' → una capa de 8 neuronas seguida de una de 4."""
    if "-" in texto:
        return tuple(int(n) for n in texto.split("-"))
    n_capas, n_neuronas = (int(v) for v in texto.lower().split("x"))
    return (n_neuronas,) * n_capas

# Configuración de partida (baseline). El modelo evoluciona desde aquí, un paso a la vez.
BASELINE = dict(
    img_size=64,                      # resolución de entrada (64x64)
    datos="original",                 # conjunto de entrenamiento: "original" o "balanceado" (sección 7.3)
    capas=(10,),                      # 1 capa oculta de 10 neuronas
    activacion="relu",
    inicializador="glorot_uniform",   # inicialización de pesos (Glorot/Xavier, valor por defecto de Keras)
    dropout=0.0,
    l2=0.0,
    batch_norm=False,
    optimizador="adam",
    learning_rate=1e-3,
    perdida="cce",                    # entropía cruzada categórica
    batch_size=32,
    max_epocas=200,
    early_stopping=True,
)
PACIENCIA = 20         # épocas sin mejorar val_loss antes de detener el entrenamiento
UMBRAL_GAP = 0.10      # diferencia máxima train-val de accuracy para considerar que NO hay sobreajuste
MIN_MEJORA = 0.005     # mejora mínima de F1 macro para adoptar un cambio

def crear_optimizador(nombre, lr):
    return {"adam": lambda: keras.optimizers.Adam(lr),
            "sgd": lambda: keras.optimizers.SGD(lr),
            "sgd_momentum": lambda: keras.optimizers.SGD(lr, momentum=0.9),
            "rmsprop": lambda: keras.optimizers.RMSprop(lr)}[nombre]()

def crear_perdida(nombre):
    return {"cce": lambda: keras.losses.CategoricalCrossentropy(),
            "mse": lambda: keras.losses.MeanSquaredError(),
            "focal": lambda: keras.losses.CategoricalFocalCrossentropy(gamma=2.0)}[nombre]()

def construir_mlp(n_entradas, capas, activacion, inicializador, dropout, l2, batch_norm, optimizador,
                  learning_rate, perdida, seed=None, **_):
    reiniciar_semillas(seed)                  # misma arquitectura → mismos pesos iniciales
    reg = regularizers.l2(l2) if l2 > 0 else None
    entrada = keras.Input(shape=(n_entradas,))
    x = entrada
    for n in capas:                           # capas ocultas
        x = layers.Dense(n, kernel_regularizer=reg, kernel_initializer=inicializador)(x)
        if batch_norm:
            x = layers.BatchNormalization()(x)
        x = layers.Activation(activacion)(x)
        if dropout > 0:
            x = layers.Dropout(dropout, seed=SEED if seed is None else seed)(x)
    salida = layers.Dense(N_CLASES, activation="softmax")(x)   # 10 probabilidades que suman 1
    modelo = keras.Model(entrada, salida)
    modelo.compile(optimizer=crear_optimizador(optimizador, learning_rate),
                   loss=crear_perdida(perdida), metrics=["accuracy"])
    return modelo

RESULTADOS, HISTORIALES, CONFIGS, MODELOS = [], {}, {}, {}
ACTUAL, ACTUAL_ID, EVOLUCION = dict(BASELINE), None, []   # modelo actual y registro de su evolución

def datos_entrenamiento(cfg):
    """Devuelve el conjunto de entrenamiento original o el balanceado con data augmentation."""
    if cfg["datos"] == "balanceado":
        return DATOS_BAL[cfg["img_size"]]["X"], y_train_bal, Y_train_bal
    return DATOS[cfg["img_size"]]["X"][0], y_train, Y_train

def ejecutar_experimento(id_exp, grupo, cambio, verbose=0, guardar=True, seed=None,
                         paciencia=None, checkpoint=None, **cambios):
    cfg = {**ACTUAL, **cambios}               # parte del modelo actual y aplica el cambio
    X_tr, y_tr, Y_tr = datos_entrenamiento(cfg)
    X_va = DATOS[cfg["img_size"]]["X"][1]
    modelo = construir_mlp(X_tr.shape[1], **cfg, seed=seed)
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=paciencia or PACIENCIA,
                                               restore_best_weights=True)] if cfg["early_stopping"] else []
    if checkpoint:                            # guarda el mejor modelo (menor val_loss) durante el entrenamiento
        callbacks.append(keras.callbacks.ModelCheckpoint(checkpoint, monitor="val_loss", save_best_only=True))
    t0 = time.time()
    h = modelo.fit(X_tr, Y_tr, validation_data=(X_va, Y_val), epochs=cfg["max_epocas"],
                   batch_size=cfg["batch_size"], shuffle=True, callbacks=callbacks, verbose=verbose)
    duracion = time.time() - t0
    if checkpoint:
        modelo = keras.models.load_model(checkpoint)   # se carga el mejor modelo guardado
    p_val = modelo.predict(X_va, batch_size=1024, verbose=0)
    p_tr = modelo.predict(X_tr, batch_size=1024, verbose=0)
    yv, yt = p_val.argmax(1), p_tr.argmax(1)
    acc_tr, acc_val = accuracy_score(y_tr, yt), accuracy_score(y_val, yv)
    capas = cfg["capas"]
    fila = dict(
        id=id_exp, grupo=grupo, cambio=cambio, img_size=cfg["img_size"], datos=cfg["datos"],
        arquitectura=f"{len(capas)}x{capas[0]}" if len(set(capas)) == 1 else "-".join(map(str, capas)),
        activacion=cfg["activacion"], inicializador=cfg["inicializador"], perdida=cfg["perdida"],
        optimizador=cfg["optimizador"], lr=cfg["learning_rate"], batch=cfg["batch_size"],
        regularizacion=((f"dropout={cfg['dropout']} " if cfg["dropout"] else "") +
                        (f"L2={cfg['l2']} " if cfg["l2"] else "") + ("BN" if cfg["batch_norm"] else "")) or "ninguna",
        epocas_ejecutadas=len(h.history["loss"]), mejor_epoca=int(np.argmin(h.history["val_loss"]) + 1),
        val_logloss=log_loss(y_val, p_val, labels=range(N_CLASES)),
        val_accuracy=acc_val,
        val_precision=precision_score(y_val, yv, average="macro", zero_division=0),
        val_recall=recall_score(y_val, yv, average="macro", zero_division=0),
        val_f1=f1_score(y_val, yv, average="macro"),
        train_accuracy=acc_tr, gap_train_val=acc_tr - acc_val,
        parametros=modelo.count_params(), segundos=round(duracion, 1))
    if guardar:
        RESULTADOS[:] = [r for r in RESULTADOS if r["id"] != id_exp] + [fila]
        HISTORIALES[id_exp], CONFIGS[id_exp], MODELOS[id_exp] = h.history, cfg, modelo
    print(f"{id_exp:<14} {cambio:<40} F1 val={fila['val_f1']:.4f}  acc train={acc_tr:.4f}  acc val={acc_val:.4f}  "
          f"gap={fila['gap_train_val']:+.4f}  épocas={fila['epocas_ejecutadas']} ({duracion:.0f}s)")
    return modelo, fila

def fila_de(id_exp):
    return next(r for r in RESULTADOS if r["id"] == id_exp)

COLS_TABLA = ["id", "cambio", "epocas_ejecutadas", "mejor_epoca", "train_accuracy", "val_accuracy",
              "val_precision", "val_recall", "val_f1", "gap_train_val", "sobreajuste", "parametros"]

def tabla(ids):
    """Tabla comparativa: mejor F1 en verde; las filas con sobreajuste se marcan en la columna 'sobreajuste'."""
    df = pd.DataFrame(RESULTADOS).set_index("id").loc[ids].reset_index()
    df["sobreajuste"] = np.where(df["gap_train_val"] > UMBRAL_GAP, "sí", "no")
    return df[COLS_TABLA].style.format({c: "{:.4f}" for c in COLS_TABLA[4:10]}) \
        .highlight_max(subset=["val_f1"], color="#c6efce") \
        .map(lambda v: "color: #c62828; font-weight: bold" if v == "sí" else "", subset=["sobreajuste"])

def graficar(ids, titulo, archivo):
    """Curvas de entrenamiento (línea continua) y validación (línea punteada) de varios experimentos."""
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    for k, i in enumerate(ids):
        h, color = HISTORIALES[i], f"C{k}"
        etiqueta = fila_de(i)["cambio"]
        ax[0].plot(h["loss"], color=color, label=f"{etiqueta} (train)")
        ax[0].plot(h["val_loss"], color=color, ls="--", label=f"{etiqueta} (val)")
        ax[1].plot(h["accuracy"], color=color, label=f"{etiqueta} (train)")
        ax[1].plot(h["val_accuracy"], color=color, ls="--", label=f"{etiqueta} (val)")
    ax[0].set_title(f"{titulo} — pérdida"); ax[1].set_title(f"{titulo} — accuracy")
    for a in ax:
        a.set_xlabel("época"); a.grid(alpha=.3)
    ax[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5))
    plt.tight_layout(); plt.savefig(f"images/{archivo}.png", dpi=150, bbox_inches="tight"); plt.show()

def curvas_train_val(id_exp, archivo=None):
    """Curvas de entrenamiento vs validación de un experimento, marcando la mejor época."""
    h = HISTORIALES[id_exp]; mejor = int(np.argmin(h["val_loss"]))
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    for a, m in zip(ax, ["loss", "accuracy"]):
        a.plot(h[m], label=f"train {m}"); a.plot(h[f"val_{m}"], ls="--", label=f"val {m}")
        a.axvline(mejor, ls=":", c="gray", label=f"mejor época ({mejor + 1})")
        a.set_title(f"{id_exp} — {m}"); a.set_xlabel("época"); a.legend(); a.grid(alpha=.3)
    plt.tight_layout()
    if archivo:
        plt.savefig(f"images/{archivo}.png", dpi=150, bbox_inches="tight")
    plt.show()

def diagnosticar(id_exp):
    """Diagnóstico: underfitting, overfitting o ajuste razonable."""
    r = fila_de(id_exp)
    if r["gap_train_val"] > UMBRAL_GAP and r["train_accuracy"] < 0.5:
        d = "underfitting y overfitting a la vez: aprende poco en train y eso poco no se traslada a validación"
    elif r["gap_train_val"] > UMBRAL_GAP:
        d = "overfitting: el modelo rinde claramente mejor en train que en validación"
    elif r["train_accuracy"] < 0.5:
        d = "underfitting: el modelo no logra aprender ni siquiera los datos de entrenamiento"
    else:
        d = "ajuste razonable"
    print(f"{id_exp}: acc train={r['train_accuracy']:.4f} | acc val={r['val_accuracy']:.4f} | "
          f"gap={r['gap_train_val']:+.4f} | mejor época {r['mejor_epoca']} de {r['epocas_ejecutadas']} → {d}")

def elegir(ids, actual_id=None):
    """Elige, entre los experimentos indicados, el de mayor F1 en validación SIN sobreajuste."""
    df = pd.DataFrame(RESULTADOS).set_index("id").loc[ids]
    validos = df[df["gap_train_val"] <= UMBRAL_GAP]
    if validos.empty:
        ganador = df["gap_train_val"].idxmin()
        motivo = "todos superan el límite de sobreajuste; se elige el de menor gap"
    else:
        mejor = validos.sort_values(["val_f1", "val_logloss"], ascending=[False, True]).index[0]
        if (actual_id in validos.index and mejor != actual_id
                and validos.loc[mejor, "val_f1"] - validos.loc[actual_id, "val_f1"] < MIN_MEJORA):
            ganador, motivo = actual_id, f"ninguna alternativa mejora el F1 en al menos {MIN_MEJORA} sin sobreajuste"
        else:
            ganador, motivo = mejor, "mayor F1 en validación entre las configuraciones sin sobreajuste"
    print(f"   → Elegido: {ganador} ({df.loc[ganador, 'cambio']}): {motivo}")
    return ganador

def avanzar(paso, ids):
    """Elige el mejor experimento del paso y lo convierte en el nuevo modelo actual."""
    global ACTUAL, ACTUAL_ID
    ganador = elegir(ids, ACTUAL_ID)
    ACTUAL, ACTUAL_ID = dict(CONFIGS[ganador]), ganador
    r = fila_de(ganador)
    EVOLUCION.append({"paso": paso, "modelo": ganador, "configuración": r["cambio"],
                      "acc train": r["train_accuracy"], "acc val": r["val_accuracy"],
                      "F1 val": r["val_f1"], "gap": r["gap_train_val"]})
    return ganador

### 8.3 Protocolo experimental: evolución iterativa del modelo

El modelo **evoluciona paso a paso**: cada experimento parte del mejor modelo obtenido hasta ese momento (**modelo actual**) y cambia **un solo aspecto**. Si una alternativa es mejor y no sobreajusta, pasa a ser el nuevo modelo actual; si no, se mantiene el anterior.

| Aspecto | Decisión |
|---|---|
| Orden de los pasos | Sigue los pasos para mejorar la generalización vistos en clases: primero se ajusta la **capacidad** de la red (resolución, **primero la cantidad de neuronas y después la cantidad de capas**) y luego los hiperparámetros que no aumentan la capacidad (datos, batch size, funciones, regularización y optimizador). |
| Comparaciones acotadas | En cada paso de arquitectura se comparan **como máximo tres alternativas**, para que la comparación sea clara. |
| Learning rate | Cada arquitectura se prueba con **su propio mejor learning rate** (0,01, 0,001 y 0,0001), porque el valor adecuado depende del tamaño de la red. |
| Criterio de elección | **Primero se revisan las curvas de entrenamiento y validación** (continua y punteada): las configuraciones cuyas curvas se separan mucho (sobreajuste) se descartan. Entre las que no sobreajustan, se elige la de mayor **F1 macro en validación**. |
| Sin sobreajuste | La diferencia entre el accuracy de entrenamiento y el de validación (gap) no supera **0,10** (10 puntos porcentuales). |
| Cuándo se adopta un cambio | Si mejora el F1 del modelo actual en al menos **0,005** sin sobreajustar. Diferencias menores se consideran ruido (ver la variabilidad por semilla, sección 8.16). |
| Early stopping | Detiene el entrenamiento si la pérdida de validación no mejora en **20 épocas** (paciencia) y restaura los pesos de la mejor época. Se usa una paciencia amplia para no cortar entrenamientos que mejoran lento o con altibajos. Máximo: 200 épocas. |
| Qué se mantiene igual | Partición train/validación, semilla (reiniciada antes de cada modelo) y preprocesamiento. |
| Test | No se usa hasta la sección 10. |

**Secuencia:** baseline → resolución → neuronas (con su mejor learning rate) → capas (con su mejor learning rate) → balance de clases → batch size → activación → inicialización → pérdida → regularización → optimizador → variabilidad por semilla → **modelo final** (sección 9) → evaluación única en test (sección 10).

### 8.4 Baseline (E0)

Modelo de partida, pequeño, para luego aumentar su capacidad de forma progresiva:

| Parámetro | Valor inicial |
|---|---|
| Resolución | 64×64, escala de grises → 4.096 entradas |
| Arquitectura | 1 capa oculta de 10 neuronas |
| Activación / inicialización | ReLU / Glorot |
| Salida y pérdida | Softmax con 10 neuronas / entropía cruzada categórica |
| Optimizador / tasa de aprendizaje | Adam / 0,001 |
| Batch size | 32 |
| Datos de entrenamiento | Originales (desbalanceados) |
| Regularización | Ninguna |
| Early stopping | Paciencia 20, máximo 200 épocas |

**Hipótesis:** con solo 10 neuronas para resumir 4.096 entradas, el modelo tendrá poca capacidad (underfitting).

In [ ]:
modelo_e0, _ = ejecutar_experimento("E0", "baseline", "Baseline (64x64, 1 capa de 10)")
modelo_e0.summary()
curvas_train_val("E0", "curvas_E0_baseline")
avanzar("Baseline", ["E0"])
diagnosticar("E0")

> **Interpretación (completar después de ejecutar)**

### 8.5 Resolución de entrada

Solo cambia el tamaño de las imágenes: 32×32 y 128×128 frente a 64×64. Más resolución conserva más detalle, pero multiplica las entradas y los parámetros de la primera capa.

In [ ]:
for t in [32, 128]:
    ejecutar_experimento(f"RES_{t}", "resolucion", f"resolución {t}x{t}", img_size=t)
IDS = ["RES_32", "E0", "RES_128"]
graficar(IDS, "Resolución de entrada", "exp_resolucion")
display(tabla(IDS))
avanzar("Resolución", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.6 Cantidad de neuronas (1 capa oculta, cada una con su mejor tasa de aprendizaje)

Antes de comparar arquitecturas, se decide **cuántas neuronas** necesita la capa oculta. Se comparan **tres cantidades**, las mismas que se usaron en los experimentos anteriores del proyecto: **10**, **30** y **100** neuronas, todas con 1 capa oculta.

Como la tasa de aprendizaje adecuada depende del tamaño de la red, **cada cantidad de neuronas se entrena con 3 tasas** (0,01, 0,001 y 0,0001) y conserva la mejor **sin sobreajuste**. Luego se comparan las tres, cada una con su mejor tasa.

**Hipótesis:** con 10 neuronas el modelo se quedará corto (underfitting); con 100 aprenderá más, pero con riesgo de sobreajuste, porque solo hay 788 imágenes de entrenamiento.

In [ ]:
NEURONAS = [10, 30, 100]
LRS = [0.01, 0.001, 0.0001]

mejores_neuronas = []
for n in NEURONAS:
    ids_n = []
    for lr in LRS:
        i = f"N{n}_lr{lr:g}"
        ejecutar_experimento(i, "neuronas", f"1 capa de {n} neuronas, lr {lr:g}", capas=(n,), learning_rate=lr)
        ids_n.append(i)
    graficar(ids_n, f"{n} neuronas: tasas de aprendizaje", f"exp_neuronas_{n}_lr")
    mejores_neuronas.append(elegir(ids_n))

# F1 en validación y gap de cada combinación de neuronas y tasa de aprendizaje
res = pd.DataFrame(RESULTADOS).set_index("id").loc[[f"N{n}_lr{lr:g}" for n in NEURONAS for lr in LRS]]
res["neuronas"] = [n for n in NEURONAS for _ in LRS]
print("\nGap entrenamiento-validación (sobre 0,10 = sobreajuste)")
display(res.pivot(index="neuronas", columns="lr", values="gap_train_val").style.format("{:+.4f}").background_gradient(cmap="Reds"))
print("F1 macro en validación (filas: neuronas; columnas: tasa de aprendizaje)")
display(res.pivot(index="neuronas", columns="lr", values="val_f1").style.format("{:.4f}").background_gradient(cmap="Greens"))

#### Las tres cantidades de neuronas, cada una con su mejor tasa de aprendizaje

In [ ]:
graficar(mejores_neuronas, "Neuronas (mejor tasa de aprendizaje de cada una)", "exp_neuronas")
display(tabla(mejores_neuronas))

prog = pd.DataFrame(RESULTADOS).set_index("id").loc[mejores_neuronas]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot([str(n) for n in NEURONAS], prog["train_accuracy"], "o-", label="accuracy entrenamiento")
ax.plot([str(n) for n in NEURONAS], prog["val_accuracy"], "o--", label="accuracy validación")
ax.plot([str(n) for n in NEURONAS], prog["val_f1"], "s-", label="F1 macro validación")
ax.set_xlabel("neuronas en la capa oculta"); ax.set_title("Efecto de la cantidad de neuronas (1 capa oculta)")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig("images/progresion_neuronas.png", dpi=150); plt.show()

avanzar("Neuronas", mejores_neuronas)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.7 Cantidad de capas (con las neuronas elegidas, cada una con su mejor tasa de aprendizaje)

Con la cantidad de neuronas elegida en la sección 8.6 (N) en la primera capa, se evalúa si conviene **agregar capas** y con cuántas neuronas. Se comparan **tres arquitecturas**, reduciendo las neuronas a la mitad en cada capa agregada:

| Arquitectura | Capas ocultas |
|---|---|
| 1 capa | N |
| 2 capas | N → N/2 |
| 3 capas | N → N/2 → N/4 |

Cada arquitectura se prueba con las 3 tasas de aprendizaje.

**Hipótesis:** más capas permiten aprender representaciones más complejas, pero aumentan el riesgo de sobreajuste y de desvanecimiento del gradiente.

In [ ]:
N_ELEGIDAS = ACTUAL["capas"][0]
CAPAS = [1, 2, 3]

def embudo(n, c):
    """c capas: la primera con n neuronas y cada una siguiente con la mitad (mínimo 4)."""
    return tuple(max(n // 2 ** k, 4) for k in range(c))

mejores_capas = [ACTUAL_ID]          # 1 capa: ya evaluada en la sección 8.6
for c in CAPAS[1:]:
    forma = embudo(N_ELEGIDAS, c)
    ids_c = []
    for lr in LRS:
        i = f"C{c}_lr{lr:g}"
        ejecutar_experimento(i, "capas", f"{c} capas {'-'.join(map(str, forma))}, lr {lr:g}", capas=forma, learning_rate=lr)
        ids_c.append(i)
    graficar(ids_c, f"{c} capas: tasas de aprendizaje", f"exp_capas_{c}_lr")
    mejores_capas.append(elegir(ids_c))

graficar(mejores_capas, "Capas (mejor tasa de aprendizaje de cada una)", "exp_capas")
display(tabla(mejores_capas))

prog = pd.DataFrame(RESULTADOS).set_index("id").loc[mejores_capas]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(CAPAS, prog["train_accuracy"], "o-", label="accuracy entrenamiento")
ax.plot(CAPAS, prog["val_accuracy"], "o--", label="accuracy validación")
ax.plot(CAPAS, prog["val_f1"], "s-", label="F1 macro validación")
ax.set_xticks(CAPAS); ax.set_xlabel("capas ocultas"); ax.set_title("Efecto de la cantidad de capas")
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.savefig("images/progresion_capas.png", dpi=150); plt.show()

avanzar("Capas", mejores_capas)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.8 Balance de clases: datos originales vs balanceados con data augmentation

Se compara el modelo actual entrenado con el conjunto original (desbalanceado) y con el conjunto balanceado + data augmentation de la sección 7.3. Ambos se evalúan en la misma validación.

**Hipótesis:** el conjunto balanceado mejorará el recall de las clases menos numerosas y, por la variedad agregada, reducirá el sobreajuste.

In [ ]:
ejecutar_experimento("DATOS_bal", "datos", "balanceado + data augmentation", datos="balanceado")
IDS = [ACTUAL_ID, "DATOS_bal"]
graficar(IDS, "Balance de clases", "exp_balance")
display(tabla(IDS))

# Recall por clase en validación: ¿mejoran las clases menos numerosas?
X_va = DATOS[ACTUAL["img_size"]]["X"][1]
recall_clase = pd.DataFrame({
    i: recall_score(y_val, MODELOS[i].predict(X_va, verbose=0).argmax(1), average=None,
                    labels=range(N_CLASES), zero_division=0) for i in IDS}, index=CLASES)
recall_clase.columns = ["original", "balanceado"]
recall_clase.plot(kind="bar", figsize=(11, 4.5), color=["#9e9e9e", "steelblue"])
plt.ylabel("recall en validación"); plt.title("Recall por clase: datos originales vs balanceados")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.savefig("images/recall_por_clase_balance.png", dpi=150); plt.show()

avanzar("Balance de clases", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.9 Tamaño del batch

Solo cambia la cantidad de imágenes por actualización de los pesos: 32, 64 y 128. Lotes pequeños dan más actualizaciones por época, pero más ruidosas; lotes grandes, menos actualizaciones pero más estables.

In [ ]:
for bs in [32, 64, 128]:
    if bs != ACTUAL["batch_size"]:
        ejecutar_experimento(f"BS_{bs}", "batch_size", f"batch size {bs}", batch_size=bs)
IDS = [ACTUAL_ID] + [f"BS_{bs}" for bs in [32, 64, 128] if bs != ACTUAL["batch_size"]]
graficar(IDS, "Tamaño del batch", "exp_batch_size")
display(tabla(IDS))
avanzar("Batch size", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.10 Épocas y early stopping

Se entrena el modelo actual durante las 200 épocas completas, **sin early stopping**, para ver qué pasa si se sigue entrenando después de la mejor época. No cambia el modelo actual: justifica el uso de early stopping con paciencia 20.

In [ ]:
ejecutar_experimento("EP_sin_ES", "epocas", "200 épocas sin early stopping", early_stopping=False, max_epocas=200)
curvas_train_val("EP_sin_ES", "curvas_sin_early_stopping")
h = HISTORIALES["EP_sin_ES"]; mejor = int(np.argmin(h["val_loss"]))
print(f"Mejor pérdida de validación en la época {mejor + 1} ({h['val_loss'][mejor]:.4f}); "
      f"en la última época: {h['val_loss'][-1]:.4f}")
print(f"Gap de accuracy en la mejor época: {h['accuracy'][mejor] - h['val_accuracy'][mejor]:+.4f} | "
      f"en la última: {h['accuracy'][-1] - h['val_accuracy'][-1]:+.4f}")

> **Interpretación (completar después de ejecutar)**

### 8.11 Funciones de activación

Solo cambia la activación de las capas ocultas (la salida sigue siendo softmax): **ReLU**, **tanh** y **sigmoide**. En clases se vio que tanh y sigmoide se saturan en los extremos (la pendiente es casi cero), lo que puede frenar el aprendizaje; por eso la sigmoide se desaconseja en capas ocultas. Se incluye para comprobarlo.

In [ ]:
for act in ["relu", "tanh", "sigmoid"]:
    if act != ACTUAL["activacion"]:
        ejecutar_experimento(f"ACT_{act}", "activacion", f"activación {act}", activacion=act)
IDS = [ACTUAL_ID] + [f"ACT_{a}" for a in ["relu", "tanh", "sigmoid"] if a != ACTUAL["activacion"]]
graficar(IDS, "Funciones de activación", "exp_activacion")
display(tabla(IDS))
avanzar("Activación", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.12 Inicialización de pesos: Glorot vs He

Glorot (Xavier) escala los pesos iniciales por √(1/k) y se recomienda para sigmoide y tanh; He los escala por √(2/k) y se recomienda para ReLU, porque ReLU anula la mitad de la señal (k = neuronas de la capa anterior). Solo cambia la inicialización.

In [ ]:
ejecutar_experimento("INIT_he", "inicializacion", "inicialización He", inicializador="he_normal")
IDS = [ACTUAL_ID, "INIT_he"]
graficar(IDS, "Inicialización de pesos: Glorot vs He", "exp_inicializacion")
display(tabla(IDS))
avanzar("Inicialización", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.13 Funciones de pérdida

Solo cambia la función de error: **entropía cruzada** (modelo actual), **error cuadrático medio (MSE)** y **focal loss**. La entropía cruzada (−log(p), con p la probabilidad de la clase correcta) castiga con fuerza las predicciones seguras y equivocadas; MSE tiene un castigo con techo bajo; focal loss multiplica la entropía cruzada por (1 − p)², concentrando el aprendizaje en las imágenes difíciles, lo que es pertinente por el desbalance.

In [ ]:
for p in ["mse", "focal"]:
    ejecutar_experimento(f"LOSS_{p}", "perdida", f"pérdida {p}", perdida=p)
IDS = [ACTUAL_ID, "LOSS_mse", "LOSS_focal"]
graficar(IDS, "Funciones de pérdida", "exp_perdida")
display(tabla(IDS))
avanzar("Pérdida", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.14 Regularización y normalización: con y sin

Se compara el modelo actual (sin regularización) con el mismo modelo agregando, de a una técnica por vez: **dropout 0,3** (apaga al azar el 30% de las neuronas en cada paso de entrenamiento), **regularización L2** (penaliza los pesos grandes) y **batch normalization** (normaliza la salida de cada capa por lote).

In [ ]:
ejecutar_experimento("REG_dropout", "regularizacion", "dropout 0.3", dropout=0.3)
ejecutar_experimento("REG_L2", "regularizacion", "L2 1e-4", l2=1e-4)
ejecutar_experimento("REG_BN", "regularizacion", "batch normalization", batch_norm=True)
IDS = [ACTUAL_ID, "REG_dropout", "REG_L2", "REG_BN"]
graficar(IDS, "Regularización y normalización: con y sin", "exp_regularizacion")
display(tabla(IDS))
avanzar("Regularización", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.15 Optimizadores

Solo cambia el optimizador, con la tasa de aprendizaje del modelo actual: **Adam**, **SGD**, **SGD con momentum** y **RMSprop**.

In [ ]:
for o in ["adam", "sgd", "sgd_momentum", "rmsprop"]:
    if o != ACTUAL["optimizador"]:
        ejecutar_experimento(f"OPT_{o}", "optimizador", f"optimizador {o}", optimizador=o)
IDS = [ACTUAL_ID] + [f"OPT_{o}" for o in ["adam", "sgd", "sgd_momentum", "rmsprop"] if o != ACTUAL["optimizador"]]
graficar(IDS, "Optimizadores", "exp_optimizador")
display(tabla(IDS))
avanzar("Optimizador", IDS)
diagnosticar(ACTUAL_ID)

> **Interpretación (completar después de ejecutar)**

### 8.16 Variabilidad por semilla

Durante todos los experimentos se usó la semilla fija 42, reiniciada antes de cada entrenamiento (sección 4.2). Aquí se mide cuánto cambia el resultado **solo por el azar** (pesos iniciales y orden de los lotes): el modelo actual se entrena con las semillas **1, 42 y 80** y se reporta el promedio y la desviación estándar del F1 en validación. No se elige la mejor semilla: el modelo se mantiene con SEED = 42.

In [ ]:
f1s = []
for s in [1, 42, 80]:
    _, fila = ejecutar_experimento(f"SEED_{s}", "semilla", f"semilla {s}", guardar=False, seed=s)
    f1s.append(fila["val_f1"])
reiniciar_semillas()
print(f"\nF1 macro de validación: {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f} (semillas 1, 42 y 80)")
print("Las diferencias entre experimentos menores que esta desviación deben interpretarse con cautela.")

> **Interpretación (completar después de ejecutar)**

### 8.17 Tabla consolidada de experimentos

Una fila por experimento con sus métricas. Se guarda en `results/experimentos.csv`. En el gráfico: en verde los modelos adoptados en algún paso y en rojo los que sobreajustan.

In [ ]:
consolidada = pd.DataFrame(RESULTADOS)
adoptados = {e["modelo"] for e in EVOLUCION}
consolidada["sobreajuste"] = np.where(consolidada["gap_train_val"] > UMBRAL_GAP, "sí", "no")
consolidada["adoptado"] = np.where(consolidada["id"].isin(adoptados), "sí", "")
consolidada.to_csv("results/experimentos.csv", index=False)

cols = ["id", "cambio", "img_size", "datos", "arquitectura", "activacion", "inicializador", "perdida", "optimizador",
        "lr", "batch", "regularizacion", "epocas_ejecutadas", "mejor_epoca", "train_accuracy", "val_accuracy",
        "val_precision", "val_recall", "val_f1", "gap_train_val", "sobreajuste", "adoptado"]
display(consolidada[cols].style.format(precision=4).background_gradient(subset=["val_f1"], cmap="Greens"))

orden = consolidada.sort_values("val_f1")
colores = ["#2e7d32" if a == "sí" else "#c62828" if s == "sí" else "#9e9e9e"
           for a, s in zip(orden["adoptado"], orden["sobreajuste"])]
plt.figure(figsize=(10, 0.3 * len(orden) + 1.5))
plt.barh(orden["id"] + " · " + orden["cambio"], orden["val_f1"], color=colores)
plt.xlabel("F1 macro (validación)"); plt.title("F1 de validación por experimento (verde: adoptado; rojo: sobreajuste)")
plt.tight_layout(); plt.savefig("images/resumen_experimentos_f1.png", dpi=150, bbox_inches="tight"); plt.show()

> **Interpretación (completar después de ejecutar)**

### 8.18 Evolución del modelo

Cómo evolucionó el modelo paso a paso: qué configuración quedó y cómo cambiaron el accuracy de entrenamiento, el de validación, el F1 y el gap. La configuración al final de esta evolución es la que se reentrena en la sección 9.

In [ ]:
evolucion = pd.DataFrame(EVOLUCION)
display(evolucion.style.format({"acc train": "{:.4f}", "acc val": "{:.4f}", "F1 val": "{:.4f}", "gap": "{:+.4f}"}))

fig, ax = plt.subplots(figsize=(11, 4.8))
x = np.arange(len(evolucion))
ax.plot(x, evolucion["acc train"], "o-", label="accuracy entrenamiento")
ax.plot(x, evolucion["acc val"], "o--", label="accuracy validación")
ax.plot(x, evolucion["F1 val"], "s-", label="F1 macro validación")
ax.set_xticks(x); ax.set_xticklabels(evolucion["paso"], rotation=30, ha="right")
ax.set_title("Evolución iterativa del modelo"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig("images/evolucion_modelo.png", dpi=150, bbox_inches="tight"); plt.show()

CONFIG_FINAL = dict(ACTUAL)
display(pd.Series(CONFIG_FINAL, name="valor").to_frame().rename_axis("parámetro"))

> **Interpretación (completar después de ejecutar)**

## 9. Modelo final

La configuración elegida en la sección 8.18 se **entrena de nuevo** con más margen:
- **Más épocas:** máximo 300.
- **Early stopping** con paciencia 30: se detiene si la pérdida de validación no mejora en 30 épocas y restaura los mejores pesos.
- **Checkpoint:** durante el entrenamiento se guarda en `models/mejor_modelo.keras` el modelo de la época con menor pérdida de validación.
- Al terminar, se **carga el mejor modelo guardado**, que es el que se evalúa en la sección 10.

In [ ]:
modelo_final, fila_final = ejecutar_experimento(
    "FINAL", "final", "modelo final (reentrenado)", paciencia=30,
    checkpoint="models/mejor_modelo.keras", **{**CONFIG_FINAL, "max_epocas": 300})
ID_FINAL = "FINAL"
curvas_train_val(ID_FINAL, "curvas_modelo_final")
diagnosticar(ID_FINAL)

with open("results/configuracion_final.json", "w") as f:
    json.dump({"seed": SEED, "clases": CLASES,
               **{k: (list(v) if isinstance(v, tuple) else v) for k, v in CONFIG_FINAL.items()}}, f, indent=2)
print("Mejor modelo: models/mejor_modelo.keras | configuración: results/configuracion_final.json")

> **Interpretación (completar después de ejecutar)**

## 10. Evaluación del mejor modelo

El mejor modelo (sección 9) se evalúa **una sola vez** sobre las imágenes de las carpetas `Test`, que no se usaron para ninguna decisión. Se muestran las métricas generales y el **reporte de clasificación** (precision, recall y F1 de cada tipo de fractura). Estos resultados no se usan para volver a modificar el modelo.

In [ ]:
X_test_final = DATOS[CONFIG_FINAL["img_size"]]["X"][2]
p_test = modelo_final.predict(X_test_final, batch_size=1024, verbose=0)
y_pred = p_test.argmax(1)            # clase con mayor probabilidad
confianza = p_test.max(1)            # probabilidad asignada a esa clase

resumen_test = pd.DataFrame({
    "Accuracy": [accuracy_score(y_test, y_pred)],
    "Precision (macro)": [precision_score(y_test, y_pred, average="macro", zero_division=0)],
    "Recall (macro)": [recall_score(y_test, y_pred, average="macro", zero_division=0)],
    "F1 (macro)": [f1_score(y_test, y_pred, average="macro")],
}, index=[f"Test — {ID_FINAL}"])
display(resumen_test.style.format("{:.4f}"))
print(classification_report(y_test, y_pred, target_names=CLASES, digits=3, zero_division=0))

> **Interpretación (completar después de ejecutar)**

### 10.1 Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=range(N_CLASES))
fig, ax = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASES, yticklabels=CLASES, ax=ax[0])
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASES, yticklabels=CLASES, ax=ax[1])
ax[0].set_title("Matriz de confusión (cantidad de imágenes)")
ax[1].set_title("Matriz normalizada por fila (= recall de cada clase)")
for a in ax:
    a.set_xlabel("Predicción"); a.set_ylabel("Clase real")
    a.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig("images/matriz_confusion_final.png", dpi=150, bbox_inches="tight"); plt.show()

#### 10.1.1 Ejemplo de cálculo de las métricas

Para mostrar de dónde salen los números, se calculan a mano las métricas de la clase con **menor F1**, leyendo directamente la matriz de confusión:
- **VP:** la celda de la diagonal de esa clase.
- **FP:** el resto de su **columna** (otras clases predichas como esta).
- **FN:** el resto de su **fila** (imágenes de esta clase predichas como otra).

In [ ]:
f1_por_clase = f1_score(y_test, y_pred, average=None, labels=range(N_CLASES), zero_division=0)
k = int(np.argmin(f1_por_clase))
vp = cm[k, k]
fp = cm[:, k].sum() - vp
fn = cm[k, :].sum() - vp
precision_k = vp / (vp + fp) if vp + fp else 0
recall_k = vp / (vp + fn) if vp + fn else 0
f1_k = 2 * precision_k * recall_k / (precision_k + recall_k) if precision_k + recall_k else 0
print(f"Clase: {CLASES[k]}")
print(f"VP = {vp} | FP = {fp} | FN = {fn}")
print(f"Precision = VP / (VP + FP) = {vp} / {vp + fp} = {precision_k:.3f}")
print(f"Recall    = VP / (VP + FN) = {vp} / {vp + fn} = {recall_k:.3f}")
print(f"F1        = 2·P·R / (P + R) = {f1_k:.3f}")
print(f"Accuracy global = aciertos / total = {np.trace(cm)} / {cm.sum()} = {np.trace(cm) / cm.sum():.3f}")

> **Interpretación (completar después de ejecutar)**

### 10.2 Validación vs test y gap train-test

Se comparan las métricas de validación (con las que se eligió el modelo) con las de test. Si son parecidas, la selección no quedó "ajustada" a la validación. El gap train-test se muestra como diagnóstico de generalización; **no** se usa para volver a modificar el modelo.

In [ ]:
X_train_final = DATOS[CONFIG_FINAL["img_size"]]["X"][0]
p_train = modelo_final.predict(X_train_final, batch_size=1024, verbose=0).argmax(1)
comparacion = pd.DataFrame({
    "train": [accuracy_score(y_train, p_train), f1_score(y_train, p_train, average="macro")],
    "validación": [fila_final["val_accuracy"], fila_final["val_f1"]],
    "test": [resumen_test.iloc[0]["Accuracy"], resumen_test.iloc[0]["F1 (macro)"]],
}, index=["accuracy", "F1 macro"])
comparacion["validación − test"] = comparacion["validación"] - comparacion["test"]
comparacion["train − test"] = comparacion["train"] - comparacion["test"]
display(comparacion.style.format("{:.4f}"))

> **Interpretación (completar después de ejecutar)**

## 11. Visualización de predicciones correctas e incorrectas

Se buscan errores **representativos**: los pares de clases más confundidos, errores de todas las clases (elegidos al azar con semilla fija, no los primeros de la lista) con la confianza de cada predicción, y los errores que el modelo cometió con más seguridad.

### 11.1 Resumen de errores y pares más confundidos

In [ ]:
err = y_pred != y_test
print(f"Test: {len(y_test)} imágenes | correctas: {(~err).sum()} ({(~err).mean():.1%}) | "
      f"incorrectas: {err.sum()} ({err.mean():.1%})\n")
por_clase = pd.DataFrame({"total": np.bincount(y_test, minlength=N_CLASES),
                          "errores": np.bincount(y_test[err], minlength=N_CLASES)}, index=CLASES)
por_clase["% error"] = (por_clase["errores"] / por_clase["total"]).map("{:.1%}".format)
display(por_clase)

pares = [(CLASES[i], CLASES[j], cm[i, j]) for i in range(N_CLASES) for j in range(N_CLASES) if i != j and cm[i, j] > 0]
display(pd.DataFrame(pares, columns=["clase real", "predicha como", "casos"])
        .sort_values("casos", ascending=False).head(8))

plt.figure(figsize=(8, 4))
if err.any():
    plt.hist(confianza[err], bins=20, alpha=.6, label="incorrectas", density=True)
if (~err).any():
    plt.hist(confianza[~err], bins=20, alpha=.6, label="correctas", density=True)
plt.xlabel("confianza (probabilidad de la clase predicha)"); plt.legend()
plt.title("Confianza de las predicciones en test")
plt.tight_layout(); plt.savefig("images/confianza_predicciones.png", dpi=150); plt.show()

> **Interpretación (completar después de ejecutar)**

### 11.2 Predicciones incorrectas: ejemplos de todas las clases y errores más confiados

In [ ]:
rng = np.random.default_rng(SEED)
idx_errores = []
for k in range(N_CLASES):
    candidatos_k = np.where(err & (y_test == k))[0]
    idx_errores += list(rng.choice(candidatos_k, size=min(2, len(candidatos_k)), replace=False))
idx_confiados = np.where(err)[0][np.argsort(-confianza[err])][:6]

def mostrar(indices, titulo, archivo):
    if len(indices) == 0:
        print("Sin casos para mostrar."); return
    columnas = min(5, len(indices)); filas_fig = int(np.ceil(len(indices) / columnas))
    fig, axes = plt.subplots(filas_fig, columnas, figsize=(3.4 * columnas, 3.8 * filas_fig), squeeze=False)
    for a in axes.ravel():
        a.axis("off")
    for a, i in zip(axes.ravel(), indices):
        a.imshow(Image.open(test_df["ruta"][i]).convert("L"), cmap="gray")
        a.set_title(f"Real: {CLASES[y_test[i]]}\nPred: {CLASES[y_pred[i]]} ({confianza[i]:.0%})", fontsize=8,
                    color="green" if y_test[i] == y_pred[i] else "red")
    fig.suptitle(titulo); plt.tight_layout()
    plt.savefig(f"images/{archivo}.png", dpi=130, bbox_inches="tight"); plt.show()

mostrar(idx_errores, "Errores de todas las clases (muestra aleatoria con semilla fija)", "errores_por_clase")
mostrar(idx_confiados, "Errores cometidos con mayor confianza", "errores_mas_confiados")

> **Interpretación (completar después de ejecutar)**

### 11.3 Predicciones correctas de todas las clases

In [ ]:
idx_aciertos = []
for k in range(N_CLASES):
    candidatos_k = np.where(~err & (y_test == k))[0]
    idx_aciertos += list(rng.choice(candidatos_k, size=min(1, len(candidatos_k)), replace=False))
mostrar(idx_aciertos, "Un acierto por clase (cuando existe)", "aciertos_por_clase")

> **Interpretación (completar después de ejecutar)**

## 12. Conclusiones y trabajos futuros

La celda siguiente resume **solo resultados obtenidos en esta ejecución**; las conclusiones escritas deben basarse en estos números.

In [ ]:
e0 = fila_de("E0")
print(f"Baseline E0 (64x64, 1 capa de 128): F1 val {e0['val_f1']:.4f}, acc train {e0['train_accuracy']:.4f}, "
      f"acc val {e0['val_accuracy']:.4f}, gap {e0['gap_train_val']:+.4f}\n")
print("Evolución del modelo (sección 8.18):")
display(pd.DataFrame(EVOLUCION).style.format({"acc train": "{:.4f}", "acc val": "{:.4f}", "F1 val": "{:.4f}", "gap": "{:+.4f}"}))
print(f"\nModelo final ({ID_FINAL}): {CONFIG_FINAL}")
print(f"Validación: F1 {fila_final['val_f1']:.4f} | Test: " +
      " | ".join(f"{c} {v:.4f}" for c, v in resumen_test.iloc[0].items()))

### Conclusiones

[COMPLETAR con base en el resumen anterior]

- **Datos:** qué mostró la exploración (desbalance, tamaños, duplicados) y qué decisiones de preparación se tomaron.
- **Punto de partida:** resultado del baseline y su diagnóstico.
- **Neuronas y capas:** cómo cambió el desempeño al variar progresivamente las neuronas y luego las capas; qué tasa de aprendizaje funcionó mejor para cada arquitectura; dónde apareció el sobreajuste.
- **Balance de clases:** si el conjunto balanceado con data augmentation mejoró el F1 y el recall de las clases menos numerosas.
- **Batch size, activación, inicialización, pérdida, regularización y optimizador:** efecto observado de cada uno y elección final.
- **Semilla:** cuánto varía el resultado por el azar y qué diferencias son confiables.
- **Modelo final y resultados en test:** qué significan para el problema (sección 3).

### Limitaciones

- **Estructura espacial:** al aplanar la radiografía, el MLP trata cada píxel como una variable independiente y no sabe qué píxeles son vecinos. Por eso no aprovecha bordes, líneas ni formas, que son justamente lo que distingue una fractura oblicua de una espiral o una fisura. Es una limitación del tipo de red, no del entrenamiento: agregar neuronas o épocas no la resuelve.
- **Detalle vs cantidad de entradas:** para conservar detalles finos se necesita más resolución, pero cada aumento multiplica las entradas y los parámetros (sección 8.5), lo que con pocas imágenes favorece el sobreajuste.
- **Datos:** unas 100 imágenes de entrenamiento por clase, con distintas zonas del cuerpo y exposiciones, es poco para 10 clases visualmente parecidas.
- [COMPLETAR con las limitaciones observadas en los resultados].

### Trabajos futuros

- **Agrupar las fracturas por zona del cuerpo** (parte superior o inferior), lo que requiere etiquetar manualmente las radiografías.
- **Pesos por clase** (`class_weight`) como alternativa al submuestreo para compensar el desbalance.
- **Validación cruzada** y repetir cada experimento con varias semillas, para obtener resultados más confiables con pocos datos.
- **Redes convolucionales (CNN) o transfer learning**, que, a diferencia del MLP, aprovechan la relación entre píxeles vecinos.

### 12.1 Entorno de ejecución

In [ ]:
import sklearn
gpus = tf.config.list_physical_devices("GPU")
print(f"Fecha de ejecución : {datetime.datetime.now():%Y-%m-%d %H:%M}")
print(f"Entorno            : {'Google Colab' if 'google.colab' in sys.modules else 'local'} | {platform.platform()}")
print(f"Python             : {platform.python_version()}")
print(f"Hardware           : {'GPU ' + str([g.name for g in gpus]) if gpus else 'CPU'}")
print(f"TensorFlow {tf.__version__} | Keras {keras.__version__} | NumPy {np.__version__} | "
      f"Pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"Tiempo total de entrenamiento de los experimentos: {sum(r['segundos'] for r in RESULTADOS) / 60:.1f} min")

## 13. Reproducción y referencias

### Cómo ejecutar este notebook

- **Repositorio:** https://github.com/Aran-Opazo/mlp_deep_learning
- **Notebook:** `notebooks/COLAB_DEEP_LEARNING_MLP.ipynb`. En Colab: *File → Open notebook → GitHub*.
- **Credenciales:** crear un token en Kaggle (*Settings → API → Create New Token*) y guardarlo en Colab como secreto con el nombre `KAGGLE_TOKEN` (panel 🔑), habilitando su acceso para este notebook.
- **Dependencias:** Colab ya las incluye. En local: `pip install -r requirements.txt`.
- **Orden de ejecución:** *Runtime → Run all*. No ejecutar celdas sueltas fuera de orden: los experimentos dependen de la partición y de la semilla definidas antes.
- **Tiempo aproximado:** [COMPLETAR con el valor de la sección 12.2]. Se recomienda GPU: *Runtime → Change runtime type*.
- **Resultados generados:** modelo en `models/modelo_mlp_final.keras`, configuración en `results/configuracion_final.json`, tabla de experimentos en `results/experimentos.csv` y figuras en `images/`.

### Referencias

- Dataset: Bone Break Classification Image Dataset — https://www.kaggle.com/datasets/pkdarabi/bone-break-classification-image-dataset
- Keras — capa Dense: https://keras.io/api/layers/core_layers/dense/
- Keras — Dropout: https://keras.io/api/layers/regularization_layers/dropout/
- Keras — BatchNormalization: https://keras.io/api/layers/normalization_layers/batch_normalization/
- Keras — regularizadores (L2): https://keras.io/api/layers/regularizers/
- Keras — funciones de activación: https://keras.io/api/layers/activations/
- Keras — inicializadores de pesos (Glorot y He): https://keras.io/api/layers/initializers/
- Keras — funciones de pérdida: https://keras.io/api/losses/
- Keras — optimizadores: https://keras.io/api/optimizers/
- Keras — EarlyStopping: https://keras.io/api/callbacks/early_stopping/
- scikit-learn — métricas de clasificación: https://scikit-learn.org/stable/modules/model_evaluation.html